In [43]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/books.db")

# create a cursor connects python to database
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

# Drop existing tables so the database can be rebuilt without duplicate data
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

# categories table
cursor.execute("""
CREATE TABLE IF NOT EXISTS  categories (
category_id INTEGER PRIMARY KEY AUTOINCREMENT,
category_name TEXT UNIQUE NOT NULL
)
""")

# create books table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        book_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        price_gbp REAL NOT NULL,
        price_inr REAL NOT NULL,
        rating INTEGER NOT NULL,
        in_stock INTEGER NOT NULL,
        category_id INTEGER NOT NULL,
        FOREIGN KEY (category_id)
            REFERENCES categories(category_id)
    )
""")
conn.commit()
# conn.close()
print("Database and tables created successfully.")


Database and tables created successfully.


In [44]:
# check all the tables

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

[('sqlite_sequence',), ('categories',), ('books',)]


In [45]:
df = pd.read_csv("data/clean_books.csv")
df.info()
categories = df["category"].unique()

print(categories)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      108 non-null    object 
 1   price_gbp  108 non-null    float64
 2   price_inr  108 non-null    float64
 3   rating     108 non-null    int64  
 4   in_stock   108 non-null    bool   
 5   category   108 non-null    object 
dtypes: bool(1), float64(2), int64(1), object(2)
memory usage: 4.4+ KB
['Travel' 'Music' 'Art' 'Horror' 'History' 'Health' 'Food and Drink'
 'Religion']


In [46]:
# insert into the category table

for category in categories:
    cursor.execute("Insert or ignore into categories (category_name) values (?)",
                   (category,))
    conn.commit()

In [47]:
# map category_ids to categories

cursor.execute("SELECT category_id, category_name FROM categories")
category_rows = cursor.fetchall()

category_map = {
    category_name: category_id
    for category_id, category_name in category_rows
}

print(category_map)

{'Travel': 1, 'Music': 2, 'Art': 3, 'Horror': 4, 'History': 5, 'Health': 6, 'Food and Drink': 7, 'Religion': 8}


In [48]:
# insert into books table

for _, row in df.iterrows():

    category_id = category_map[row["category"]]

    cursor.execute("""
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

In [49]:
cursor.execute("Select * from categories")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'Travel')
(2, 'Music')
(3, 'Art')
(4, 'Horror')
(5, 'History')
(6, 'Health')
(7, 'Food and Drink')
(8, 'Religion')


In [50]:
cursor.execute("select * from books")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, "It's Only the Himalayas", 45.17, 4765.44, 2, 1, 1)
(2, 'Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Beyond', 49.43, 5214.86, 4, 1, 1)
(3, 'See America: A Celebration of Our National Parks & Treasured Sites', 48.87, 5155.78, 3, 1, 1)
(4, 'Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel', 36.94, 3897.17, 2, 1, 1)
(5, 'Under the Tuscan Sun', 37.33, 3938.31, 3, 1, 1)
(6, 'A Summer In Europe', 44.34, 4677.87, 2, 1, 1)
(7, 'The Great Railway Bazaar', 30.54, 3221.97, 1, 1, 1)
(8, 'A Year in Provence (Provence #1)', 56.88, 6000.84, 4, 1, 1)
(9, 'The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2)', 23.21, 2448.66, 1, 1, 1)
(10, 'Neither Here nor There: Travels in Europe', 38.95, 4109.23, 3, 1, 1)
(11, '1,000 Places to See Before You Die', 26.08, 2751.44, 5, 1, 1)
(12, 'Rip it Up and Start Again', 35.02, 3694.61, 5, 1, 2)
(13, 'Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',